In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay



In [ ]:
np.random.seed(42)

n_samples = 10000  # total dataset size

data = {
    "tutor_rating": np.random.uniform(2.5, 5.0, n_samples),
    "rating_count": np.random.randint(1, 100, n_samples),
    "experience_level": np.random.randint(1, 6, n_samples),
    "sessions": np.random.randint(1, 20, n_samples),
    "avg_improvement": np.random.uniform(-1.5, 3.0, n_samples),
    "subject": np.random.choice(
        ["Math", "Physics", "English", "Computer_Science"],
        n_samples
    )
}

df = pd.DataFrame(data)




In [ ]:
def success_probability(row):
    score=((row["avg_improvement"] * 0.4) +
           (row["tutor_rating"] * 0.3) +
           (row["sessions"] * 0.2)+ 
           (row['experience_level']*0.1))
    prob=1/(1+np.exp(-(score -5)))
    return prob

In [ ]:
df["success_prob"] = df.apply(success_probability, axis=1)
df['success']=( df["success_prob"] > 0.5).astype(int)

noise_indices=np.random.choice(df.index, size=int(0.1 * len(df)), replace=False )
df.loc[noise_indices, 'success'] = 1- df.loc[noise_indices, 'success']
    

print(df.head())

In [ ]:
df=pd.get_dummies(df)

In [ ]:
X= df.drop(columns=['success'])
y= df['success']

In [ ]:
# for documentation purposes
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight = 'balanced')
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
feat_imp=pd.DataFrame({"Features": X_train.columns, "Importance":model.feature_importances_}).sort_values(by='Importance', ascending=False)

In [ ]:
feat_imp

In [ ]:
df.head()

In [ ]:
scores=cross_val_score(model,X_train, y_train, cv=5)
print('Cross-validation scores',scores)
print('Mean CV accuracy', scores.mean())

In [ ]:
param_grid = {'n_estimators':[50,100,200], 'max_depth':[None, 5,10,20], 'min_samples_split':[2,5,10]}
GS= GridSearchCV(model, param_grid, cv=5, scoring='accuracy',n_jobs=1)
GS.fit(X_train, y_train)
print('Best Parameters:',GS.best_params_)
print('Best CV Score:',GS.best_score_)

In [ ]:
best_model= GS.best_estimator_
y_pred= best_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
df['success'].value_counts()


In [ ]:
joblib.dump (best_model, "tutoring_model.pkl")


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
play = ConfusionMatrixDisplay(confusion_matrix=cm)
play.plot()
plt.show